In [1]:
def add_run(run, log=[]):
    log.append(run)
    return log


print(add_run("a"))
print(add_run("b"))

['a']
['a', 'b']


In [1]:
import dis

dis.dis("total = total + 1")

  0           RESUME                   0

  1           LOAD_NAME                0 (total)
              LOAD_SMALL_INT           1
              BINARY_OP                0 (+)
              STORE_NAME               0 (total)
              LOAD_CONST               1 (None)
              RETURN_VALUE


In [6]:
import timeit

setup = "data = range(-1000, 1000)"
print("genexp:", timeit.timeit("sum(abs(x) for x in data)", setup, number=10_000))
print("map:   ", timeit.timeit("sum(map(abs, data))",       setup, number=10_000))

genexp: 0.506784625002183
map:    0.3185913749912288


In [2]:
import sys

print(sys.getsizeof(1))
print(sys.getsizeof([]))

28
56


In [1]:
a = [1, 2, 3]     # refcount 1
b = a             # refcount 2   <- no copy happened
del a             # refcount 1

In [2]:
print(b)

[1, 2, 3]


In [3]:
a = [1, 2, 3]
b = a
b.append(4)
print(a)          # [1, 2, 3, 4]
print(a is b)     # True   — same object
print(id(a), id(b))

[1, 2, 3, 4]
True
4460055424 4460055424


In [6]:
x = 4
y = 4
print(x == y)      # True   — same value    (calls __eq__)
print(x is y)      # False  — different objects (compares identity)

True
True


In [7]:
id(x)

4350412112

In [8]:
id(y)

4350412112

In [ ]:
grid = [[0] * 3] * 3
grid[0][0] = 1
print(grid)        # [[1, 0, 0], [1, 0, 0], [1, 0, 0]]

[[1, 0, 0], [1, 0, 0], [1, 0, 0]]


In [13]:
grid, grid[0][0]

([[0, 0, 0], [0, 0, 0], [0, 0, 0]], 0)

In [ ]:
import copy

runs = [{"acc": 0.9, "tags": ["v1"]}]
shallow = list(runs)              # also: runs[:], copy.copy(runs), runs.copy()
deep = copy.deepcopy(runs)

runs[0]["tags"].append("v2")
shallow[0]["tags"]                # ['v1', 'v2']  — shared
deep[0]["tags"]                   # ['v1']

In [22]:
list_a = [30, 40]
t = (1, 2, list_a)
print(t)
list_a += [50, 60]
print(t)

(1, 2, [30, 40])
(1, 2, [30, 40, 50, 60])


In [24]:
hash((1, 2, [3]))

TypeError: unhashable type: 'list'

In [25]:
"run-1,xgboost,0.001,64,0.83".split(",")

['run-1', 'xgboost', '0.001', '64', '0.83']

In [33]:
run_id, *_, numbers = "run-1,xgboost,0.001,64,0.83".split(",")

In [34]:
run_id, numbers

('run-1', '0.83')

In [ ]:
*args, **kwargs

In [ ]:
for run_id, (accuracy, duration) in results:

In [41]:
# ---------------------------------------------------------------------------
# TASK 1 — Aliasing
#
# A colleague wrote:
#
#     grid = [[0] * cols] * rows
#
# and their confusion matrix updates all rows at once. Write a version that
# does not have that bug. One line is enough.
# ---------------------------------------------------------------------------

def build_grid(rows: int, cols: int) -> list[list[int]]:
    """Return a rows x cols grid of zeros whose rows are independent objects."""
    return [[0] * cols for _ in range(rows)]

build_grid(3, 3)

[[0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [45]:


# ---------------------------------------------------------------------------
# TASK 2 — Unpacking with a star target
#
# Raw line:  'run-00003,xgboost,titanic,0.001,64,0.8312,131.4'
#
# Parse it into a tuple with the right types:
#   (str, str, str, float, int, float, float)
#
# Constraint: use *starred unpacking* to split the line into
# "identity fields" and "numeric fields" instead of indexing seven times.
# ---------------------------------------------------------------------------


def to_record(line: str) -> tuple:
    """Parse one raw CSV line into a properly typed tuple."""
    run_id, model, dataset, *numbers = line.split(",")
    lr, batch, acc, duration = numbers
    return(run_id, model, dataset, float(lr), int(batch), float(acc), float(duration))

to_record('run-00003,xgboost,titanic,0.001,64,0.8312,131.4')

('run-00003', 'xgboost', 'titanic', 0.001, 64, 0.8312, 131.4)

In [ ]:
from common import ACCURACY, DURATION
# ---------------------------------------------------------------------------
# TASK 3 — Sorting with a key
#
# Return the k runs with the highest accuracy, best first.
# Ties are broken by the *shorter* duration.
#
# Constraint: no manual loops over a sorted copy, and do not sort twice.
# Hint: operator.itemgetter, and think about what a tuple key does.
# ---------------------------------------------------------------------------


def top_k(runs: list[tuple], k: int) -> list[tuple]:
    """Return the k best runs: highest accuracy first, shortest duration wins ties."""
    return sorted(runs, key=lambda r:(r[ACCURACY], -r[DURATION]), reverse=True)[:k]

In [47]:
from collections import deque

# ---------------------------------------------------------------------------
# TASK 4 — A bounded buffer
#
# Compute the trailing moving average of `values` over `window` points.
# Output has the same length as the input; for the first window-1 positions,
# average over whatever is available so far.
#
# Constraint: use collections.deque with maxlen. Do NOT slice the list
# (values[i-window:i]) — we want O(1) work per point, not O(window).
# ---------------------------------------------------------------------------


def moving_average(values: list[float], window: int) -> list[float]:
    """Trailing moving average, one output value per input value."""
    buf: deque[float] = deque(maxlen=window)
    out = []
    for value in values:
        buf.append(value)
        out.append(sum(buf) / len(buf))
    return out

In [48]:
class Key:
    def __init__(self, tags):
        self.tags = tags

    def __hash__(self):
        return hash(tuple(self.tags))

    def __eq__(self, other):
        return self.tags == other.tags


k = Key(["a"])
s = {k}
k.tags.append("b")
print(k in s)  # False
print(list(s)[0] is k)  # True   — it is in there, and unreachable


False
True


In [49]:
old = {"lr": 0.001, "batch": 64, "dropout": 0.1}
new = {"lr": 0.001, "batch": 128, "weight_decay": 0.0}

new.keys() - old.keys()      # {'weight_decay'}   added
old.keys() - new.keys()      # {'dropout'}        removed
old.keys() & new.keys()      # {'lr', 'batch'}    common

{'batch', 'lr'}

In [56]:
set_keys = set(new.keys())
set_keys.add("something")
set_keys

{'batch', 'lr', 'something', 'weight_decay'}

In [ ]:
a & b    # intersection      a | b    # union
a - b    # difference        a ^ b    # symmetric difference

In [ ]:
from typing import NamedTuple
from dataclasses import dataclass

class Run(NamedTuple):
    run_id: str
    model: str
    accuracy: float

@dataclass(frozen=True, slots=True)
class RunKey:
    model: str
    dataset: str
    lr: float

In [62]:
import functools, time

def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        t0 = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            print(f"{func.__name__}: {time.perf_counter() - t0:.4f}s")
    return wrapper


@timed
def adding_list():
    [i for i in range(1000000)]
    return []

@timed
def substracting_list():
    [i for i in range(1000000)]
    return []

substracting_list()

substracting_list: 0.0276s


[]

In [ ]:
from functools import cache          # or lru_cache(maxsize=1024)

@cache
def expensive_feature(user_id: int) -> float:
    ...

In [ ]:
def parse(lines: list[str]):
    for line in lines:
        if line.startswith("#"):
            continue
        yield tuple(line.split(","))


lines = [
    "# id,name,role",
    "1,Ada,engineer",
    "2,Linus,maintainer",
    "# comment in the middle",
    "3,Grace,admiral",
]

for row in parse(lines):
    print(row)


('1', 'Ada', 'engineer')
('2', 'Linus', 'maintainer')
('3', 'Grace', 'admiral')


In [67]:
parse(lines)

<generator object parse at 0x10f0565c0>

In [72]:
import sys

sys.getsizeof([x * x for x in range(100000000)])

835128600

In [ ]:
sum([x * x for x in range(10000000)])

333333283333335000000

In [70]:
sum(x * x for x in range(10000000)) 

333333283333335000000